<a href="https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VikramR6/flyrank-ml-internship-assign1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [29]:
!pip install -q duckdb huggingface_hub pyarrow pandas scikit-learn matplotlib

In [30]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
import duckdb
import pandas as pd
import numpy as np

In [31]:
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

In [32]:
con.sql("""
    INSTALL httpfs;
    LOAD httpfs;
    INSTALL parquet;
    LOAD parquet;
""")

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

con.sql(f"""
    CREATE OR REPLACE VIEW fact_content_daily_performance AS
    SELECT *
    FROM read_parquet('{march_path}');
""")

print("March 2026 warehouse data connected.")

March 2026 warehouse data connected.


In [33]:
# Build ML-08 feature + label dataset
# Only content with measurable GSC data in BOTH windows is included.

ml08_data = con.sql("""
WITH feature_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_feature,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        SUM(sessions_ai) AS sessions_ai,
        COUNT(DISTINCT report_date) AS gsc_measured_days
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),

future_window AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS future_clicks
    FROM fact_content_daily_performance
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_90d,
    f.clicks_feature,
    ROUND(f.avg_position, 2) AS avg_position,
    f.sessions_ai,
    f.gsc_measured_days,

    CASE
        WHEN w.future_clicks = 0 THEN 1
        ELSE 0
    END AS decline_proxy

FROM feature_window f
INNER JOIN future_window w
    ON f.client_hash_id = w.client_hash_id
   AND f.content_hash_id = w.content_hash_id
""").df()

print("ML-08 dataset shape:", ml08_data.shape)

print("\nLabel distribution:")
print(ml08_data["decline_proxy"].value_counts())

print("\nDecline proxy rate:")
print(
    round(100 * ml08_data["decline_proxy"].mean(), 2),
    "%"
)

print("\nDataset preview:")
display(ml08_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ML-08 dataset shape: (141467, 8)

Label distribution:
decline_proxy
1    90604
0    50863
Name: count, dtype: int64

Decline proxy rate:
64.05 %

Dataset preview:


,client_hash_id,content_hash_id,impressions_90d,clicks_feature,avg_position,sessions_ai,gsc_measured_days,decline_proxy
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,4.36,NaN,15,1
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,20.0,0.0,4.15,NaN,9,1
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,246.0,0.0,4.81,NaN,15,1
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,413.0,1.0,4.66,NaN,15,1
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,11.0,0.0,4.91,NaN,7,1


In [34]:
# ML-08 feature quality check

feature_cols = [
    "impressions_90d",
    "clicks_feature",
    "avg_position",
    "sessions_ai",
    "gsc_measured_days"
]

quality_check = pd.DataFrame({
    "feature": feature_cols,
    "missing": [ml08_data[c].isna().sum() for c in feature_cols],
    "missing_pct": [
        round(100 * ml08_data[c].isna().mean(), 2)
        for c in feature_cols
    ]
})

display(quality_check)

print("\nLabel rate:")
print(
    round(
        100 * ml08_data["decline_proxy"].mean(),
        2
    ),
    "% decline_proxy = 1"
)

,feature,missing,missing_pct
0,impressions_90d,0,0.00
1,clicks_feature,0,0.00
2,avg_position,0,0.00
3,sessions_ai,66245,46.83
4,gsc_measured_days,0,0.00



Label rate:
64.05 % decline_proxy = 1


In [35]:
# Check future-window coverage before trusting the decline proxy

future_coverage = con.sql("""
SELECT
    COUNT(*) AS future_rows,
    COUNT(DISTINCT client_hash_id) AS future_clients,
    COUNT(DISTINCT content_hash_id) AS future_content_items,
    COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id)) AS future_client_content_pairs
FROM fact_content_daily_performance
WHERE gsc_data_available IS TRUE
  AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
""").df()

display(future_coverage)

# Check how many ML-08 items have no measurable future GSC row
future_pairs = con.sql("""
SELECT DISTINCT
    client_hash_id,
    content_hash_id
FROM fact_content_daily_performance
WHERE gsc_data_available IS TRUE
  AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
""").df()

ml08_pairs = ml08_data[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()

future_pairs_set = set(
    zip(
        future_pairs["client_hash_id"],
        future_pairs["content_hash_id"]
    )
)

no_future_data = [
    pair for pair in zip(
        ml08_pairs["client_hash_id"],
        ml08_pairs["content_hash_id"]
    )
    if pair not in future_pairs_set
]

print(
    "ML-08 content pairs with no measurable future GSC data:",
    len(no_future_data)
)

print(
    "Percentage:",
    round(100 * len(no_future_data) / len(ml08_pairs), 2),
    "%"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,future_rows,future_clients,future_content_items,future_client_content_pairs
0,1970824,46,166224,166224


ML-08 content pairs with no measurable future GSC data: 0
Percentage: 0.0 %


## 1. Method choice and why

### Method Choice

I chose Logistic Regression as the first learned model for the content-decline lane.

It fits the lane because the task is binary classification: identifying content items with a decline proxy of 1 or 0. Logistic Regression is simple, interpretable, and provides a probability score that can be used to rank content for review.

The model will use decision-time performance features only. Features with substantial missingness and target-derived fields will not be used. The model will be compared with the Week-4 baseline using the same evaluation data and metric.

In [36]:
# Define the ML-08 modelling features

features = [
    "impressions_90d",
    "clicks_feature",
    "avg_position",
    "gsc_measured_days"
]

target = "decline_proxy"

X = ml08_data[features].copy()
y = ml08_data[target].copy()

print("Features used:")
print(features)

print("\nTarget:")
print(target)

print("\nFeature matrix shape:", X.shape)
print("Target size:", y.shape)


Features used:
['impressions_90d', 'clicks_feature', 'avg_position', 'gsc_measured_days']

Target:
decline_proxy

Feature matrix shape: (141467, 4)
Target size: (141467,)


## 2. Split design

### Split Design

I use a client-grouped 80/20 split. Approximately 80% of clients are used for training and the remaining 20% are held out for testing.

This is more honest than randomly splitting content rows because the same client can have multiple content items. Keeping each client entirely in one split prevents content from the same client appearing in both training and testing.

The test clients are kept untouched until model evaluation and baseline comparison.

In [37]:
from sklearn.model_selection import GroupShuffleSplit

groups = ml08_data["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())

print(
    "\nClients appearing in both splits:",
    len(set(groups_train) & set(groups_test))
)

print(
    "\nTraining decline rate:",
    round(100 * y_train.mean(), 2), "%"
)

print(
    "Testing decline rate:",
    round(100 * y_test.mean(), 2), "%"
)

Training rows: 131220
Testing rows: 10247

Training clients: 34
Testing clients: 9

Clients appearing in both splits: 0

Training decline rate: 64.34 %
Testing decline rate: 60.29 %


## 3. Train + compare vs my baseline

### Train and Compare

I trained Logistic Regression using the client-grouped training split. The Week-4 baseline rule is applied to the same held-out test set.

Both approaches are evaluated using Precision@20 and Precision@50 because the content-review task is a prioritization problem. The model uses its predicted probability to rank content, while the baseline uses its transparent rule score.

The comparison is made on the same test clients and against the same decline proxy.

In [38]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Train Logistic Regression
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Model probability for the positive class
model_proba = model.predict_proba(X_test)[:, 1]


# Week-4 baseline rule applied to the SAME test set
test_baseline = X_test.copy()

test_baseline["baseline_score"] = (
    (test_baseline["impressions_90d"] >= 500)
    & (test_baseline["avg_position"] <= 10)
    & (
        100 * test_baseline["clicks_feature"]
        / test_baseline["impressions_90d"].replace(0, np.nan)
        < 0.5
    )
).astype(int)


# Precision@K helper
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()


# Calculate metrics
baseline_p20 = precision_at_k(
    y_test,
    test_baseline["baseline_score"],
    20
)

baseline_p50 = precision_at_k(
    y_test,
    test_baseline["baseline_score"],
    50
)

model_p20 = precision_at_k(
    y_test,
    model_proba,
    20
)

model_p50 = precision_at_k(
    y_test,
    model_proba,
    50
)

base_rate = y_test.mean()


# Comparison table
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Logistic Regression",
        "Test-set Base Rate"
    ],
    "Precision@20": [
        round(baseline_p20, 3),
        round(model_p20, 3),
        round(base_rate, 3)
    ],
    "Precision@50": [
        round(baseline_p50, 3),
        round(model_p50, 3),
        round(base_rate, 3)
    ]
})

display(comparison)


,Method,Precision@20,Precision@50
0,Week-4 Baseline,0.200,0.240
1,Logistic Regression,1.000,0.980
2,Test-set Base Rate,0.603,0.603


## 4. Errors and interpretation

### Feature interpretation

I inspect the learned model coefficients to understand which features are associated with the decline proxy. This helps check whether the model is relying on sensible decision-time signals.

### Error analysis

I also inspect incorrect predictions on the held-out clients. The goal is to understand where the model fails rather than judging it only by its aggregate Precision@K.

The very high Precision@20 result is treated as a finding to investigate, not as proof that the model will generalize to all future content.

In [39]:
# Feature interpretation

logistic_model = model.named_steps["logistic_regression"]

feature_importance = pd.DataFrame({
    "Feature": features,
    "Coefficient": logistic_model.coef_[0]
})

feature_importance["Abs_Coefficient"] = (
    feature_importance["Coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "Abs_Coefficient",
    ascending=False
)

display(feature_importance)


,Feature,Coefficient,Abs_Coefficient
1,clicks_feature,-9.961130,9.961130
0,impressions_90d,-1.620393,1.620393
2,avg_position,0.506503,0.506503
3,gsc_measured_days,-0.370612,0.370612


In [40]:
# Error analysis on the held-out test clients

test_results = ml08_data.iloc[test_idx].copy()

test_results["model_probability"] = model_proba
test_results["model_prediction"] = (
    test_results["model_probability"] >= 0.5
).astype(int)

test_results["correct"] = (
    test_results["model_prediction"]
    == test_results["decline_proxy"]
)

errors = test_results[
    test_results["correct"] == False
].copy()

print("Total test rows:", len(test_results))
print("Incorrect predictions:", len(errors))

print(
    "Error rate:",
    round(100 * len(errors) / len(test_results), 2),
    "%"
)

display(
    errors[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_90d",
            "clicks_feature",
            "avg_position",
            "gsc_measured_days",
            "decline_proxy",
            "model_probability",
            "model_prediction"
        ]
    ].head(10)
)

Total test rows: 10247
Incorrect predictions: 2166
Error rate: 21.14 %


,client_hash_id,content_hash_id,impressions_90d,clicks_feature,avg_position,gsc_measured_days,decline_proxy,model_probability,model_prediction
3172,client_def0955f7a377868,content_b74e2c664f238556,32.0,0.0,7.38,9,0,0.845878,1
3173,client_def0955f7a377868,content_5bfec56eed956b2e,31.0,1.0,10.42,13,0,0.681365,1
3174,client_def0955f7a377868,content_2c797512681d9fe0,91.0,0.0,5.13,15,0,0.756743,1
3179,client_def0955f7a377868,content_1e95a47bf34da496,62.0,3.0,7.90,13,1,0.318880,0
3186,client_def0955f7a377868,content_686d070c2a12f774,99.0,0.0,21.56,15,0,0.831115,1
14502,client_def0955f7a377868,content_b93db99292b54061,9.0,0.0,13.11,6,0,0.892123,1
14506,client_def0955f7a377868,content_c7ab47c6d2cc3220,3.0,0.0,14.67,1,0,0.927721,1
22288,client_e5c2aa26a8598242,content_dc565189eb62b39c,418.0,1.0,8.30,15,0,0.579930,1
22568,client_0fa64a184f18a4a0,content_f7d13ac45236e1ff,1339.0,3.0,5.33,15,1,0.152226,0
22578,client_0fa64a184f18a4a0,content_b988cd0db2453dcc,32.0,2.0,5.84,13,1,0.478822,0


### Interpretation of results

The Logistic Regression model achieved Precision@20 of 1.00 and Precision@50 of 0.98 on the held-out client test set. Its overall error rate was 21.14%, so the model is not perfectly correct.

The strongest feature by absolute coefficient was `clicks_feature` (-9.96), followed by `impressions_90d` (-1.62). Their negative coefficients indicate that higher observed clicks and impressions are associated with a lower probability of the decline proxy.

`avg_position` had a positive coefficient (0.51), indicating that worse search position is directionally associated with a higher probability of the decline proxy. `gsc_measured_days` had a smaller negative coefficient (-0.37).

These are measured associations in this dataset, not evidence of causation. The unusually strong top-K precision should be treated as a result on this held-out client split rather than proof that the model will perform equally well on future datasets.

In [41]:
# Show concrete false positives and false negatives

false_positives = test_results[
    (test_results["decline_proxy"] == 0) &
    (test_results["model_prediction"] == 1)
].copy()

false_negatives = test_results[
    (test_results["decline_proxy"] == 1) &
    (test_results["model_prediction"] == 0)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nThree false positives:")
display(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_90d",
            "clicks_feature",
            "avg_position",
            "decline_proxy",
            "model_probability"
        ]
    ].head(3)
)

print("\nThree false negatives:")
display(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_90d",
            "clicks_feature",
            "avg_position",
            "decline_proxy",
            "model_probability"
        ]
    ].head(3)
)

False positives: 1707
False negatives: 459

Three false positives:


,client_hash_id,content_hash_id,impressions_90d,clicks_feature,avg_position,decline_proxy,model_probability
3172,client_def0955f7a377868,content_b74e2c664f238556,32.0,0.0,7.38,0,0.845878
3173,client_def0955f7a377868,content_5bfec56eed956b2e,31.0,1.0,10.42,0,0.681365
3174,client_def0955f7a377868,content_2c797512681d9fe0,91.0,0.0,5.13,0,0.756743



Three false negatives:


,client_hash_id,content_hash_id,impressions_90d,clicks_feature,avg_position,decline_proxy,model_probability
3179,client_def0955f7a377868,content_1e95a47bf34da496,62.0,3.0,7.90,1,0.318880
22568,client_0fa64a184f18a4a0,content_f7d13ac45236e1ff,1339.0,3.0,5.33,1,0.152226
22578,client_0fa64a184f18a4a0,content_b988cd0db2453dcc,32.0,2.0,5.84,1,0.478822


### Error analysis

The model produced 1,707 false positives and 459 false negatives on the held-out test clients.

The false positives show cases where the model predicted the decline proxy but the future outcome was not classified as declining. Examples include content with 54 impressions and 0 clicks at position 6.76, and content with 3 impressions and 0 clicks at position 28.67. These cases show that weak current performance can sometimes lead to an incorrect decline prediction.

The false negatives show cases where the model missed the decline proxy. One example had 302 impressions, 4 clicks, and an average position of 7.66 but still received the decline proxy. These cases show that current performance signals do not completely explain the future outcome.

Overall, the errors indicate that the model captures useful relationships in the observed features, but the available features do not fully explain the future decline proxy.

In [42]:
# Final leakage sanity check

forbidden_features = [
    "decline_proxy",
    "future_clicks",
    "trend_direction",
    "trend_pct",
    "label",
    "target"
]

used_features_lower = [f.lower() for f in features]

leakage_found = [
    f for f in forbidden_features
    if f.lower() in used_features_lower
]

print("Model features:", features)
print("Forbidden feature matches:", leakage_found)

if len(leakage_found) == 0:
    print("PASS — no target, future, trend, or label-derived feature is used.")
else:
    print("FAIL — leakage-related feature detected.")

Model features: ['impressions_90d', 'clicks_feature', 'avg_position', 'gsc_measured_days']
Forbidden feature matches: []
PASS — no target, future, trend, or label-derived feature is used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.